In [8]:
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')  # Remove this line if running interactively
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.validation import make_valid
from shapely.geometry import MultiPolygon, Polygon
from shapely.affinity import translate

In [9]:
# ════════════════════════════════════════════════════════════════════
# STEP 0: Fix antimeridian wrapping (Russia, Fiji, etc.)
# ════════════════════════════════════════════════════════════════════

def unwrap_ring(coords):
    """Unwrap longitudes so consecutive vertices stay close to each other."""
    new_coords = []
    prev_x = None
    for x, y in coords:
        nx = x
        if prev_x is not None:
            while nx - prev_x > 180:
                nx -= 360
            while nx - prev_x < -180:
                nx += 360
        new_coords.append((nx, y))
        prev_x = nx
    return new_coords


def polygon_parts(geom):
    if geom.is_empty:
        return []
    if geom.geom_type == 'Polygon':
        return [geom]
    if geom.geom_type == 'MultiPolygon':
        return list(geom.geoms)
    if geom.geom_type == 'GeometryCollection':
        parts = []
        for g in geom.geoms:
            parts.extend(polygon_parts(g))
        return parts
    return []


def fix_geometry(geom):
    """Repair invalid polygons and correctly handle antimeridian-crossing shapes."""
    geom = make_valid(geom)
    out = []
    world_box = Polygon([(-180, -90), (180, -90), (180, 90), (-180, 90)])

    for poly in polygon_parts(geom):
        # Most polygons need no special treatment
        if (poly.bounds[2] - poly.bounds[0]) <= 180:
            out.append(poly)
            continue

        # Unwrap the polygon into a continuous longitude domain
        ext = unwrap_ring(list(poly.exterior.coords))
        holes = [unwrap_ring(list(r.coords)) for r in poly.interiors]
        unwrapped = make_valid(Polygon(ext, holes))

        for part in polygon_parts(unwrapped):
            # Intersect the part with the normal world extent, and also with
            # +/-360 shifted copies. This preserves pieces around the dateline
            # without creating long horizontal bands across the map.
            candidates = [
                part,
                translate(part, xoff=-360),
                translate(part, xoff=360),
            ]
            for cand in candidates:
                clipped = make_valid(cand.intersection(world_box))
                out.extend([p for p in polygon_parts(clipped) if p.area > 0])

    # Drop duplicates created by overlapping shifted copies
    dedup = []
    seen = set()
    for poly in out:
        key = poly.wkb
        if key not in seen:
            seen.add(key)
            dedup.append(poly)

    if not dedup:
        return geom
    if len(dedup) == 1:
        return dedup[0]
    return MultiPolygon(dedup)

In [10]:
# ════════════════════════════════════════════════════════════════════
# STEP 1: EDIT THIS — Define your country groups and colors
# ════════════════════════════════════════════════════════════════════

# Map: group_name -> list of country names (must match Natural Earth names)
# Run `print(sorted(world['name'].unique()))` to see all available names
category_map = {
    'English': [
        'United States of America', 'Canada', 'United Kingdom', 'Australia',
        'New Zealand', 'Ireland', 'South Africa', 'Jamaica', 'Barbados', 'Trinidad and Tobago',
        'Bahamas', 'Antigua and Barb.', 'Saint Lucia', 'Dominica', 'Grenada', 'Singapore'
    ],
    
    'Chinese': [
        'China', 'Taiwan', 'Hong Kong', 'Macau'
    ],
    
    'Spanish': [
        'Spain', 'Argentina', 'Bolivia', 'Chile', 'Colombia', 'Costa Rica', 'Cuba', 
        'Dominican Rep.', 'Ecuador', 'El Salvador', 'Guatemala', 'Honduras', 'Mexico', 
        'Nicaragua', 'Panama', 'Paraguay', 'Peru', 'Uruguay', 'Eq. Guinea', 'Venezuela'
    ],
    
    'Japanese': ['Japan'],
    
    'German': [
        'Germany', 'Austria', 'Switzerland', 'Liechtenstein'
    ],
    
    'Russian': ['Russia', 'Belarus'],
    
    'French': [
        'France', 'Monaco', 'Benin', 'Burkina Faso', 'Burundi', 'Central African Rep.', 
        'Chad', 'Congo', 'Dem. Rep. Congo', "Côte d'Ivoire", 'Djibouti', 'Gabon', 'Guinea', 'Mali', 
        'Niger', 'Rwanda', 'Senegal', 'Togo', 'Haiti', 'Vanuatu'
    ],
    
    'Norwegian': ['Norway'],
    
    'Portuguese': [
        'Brazil', 'Portugal', 'Angola', 'Mozambique', 'Cabo Verde', 'Guinea-Bissau', 'São Tomé and Principe', 'Timor-Leste'
    ],
}

# Colors for each group
colors = {
    'English':    '#ADD8E6',  # Light Blue
    'Chinese':    '#FF0000',  # Bright Red
    'Spanish':    '#B19CD9',  # Light Purple
    'Japanese':   '#FFFF00',  # Bright Yellow
    'German':     '#ED7D31',  # Orange
    'Russian':    '#FFF2CC',  # Light Yellow
    'French':     '#90EE90',  # Light Green
    'Norwegian':  '#8B4513',  # Brown
    'Portuguese': '#FF69B4',  # Pink
}

In [11]:
# ════════════════════════════════════════════════════════════════════
# STEP 2: EDIT THIS — Country labels (name -> display_label, x, y, fontsize)
# ════════════════════════════════════════════════════════════════════

label_map = {
    # 'United States of America': ('UNITED STATES', -100, 40, 7.5),
    # 'Canada':    ('CANADA', -100, 58, 7.5),
    # 'Brazil':    ('BRAZIL', -53, -10, 8.5),
    # 'Russia':    ('RUSSIA', 90, 62, 8.5),
    # 'China':     ('CHINA', 103, 36, 8.5),
    # 'Australia': ('AUSTRALIA', 134, -25, 7.5),
    # # 'India':     ('INDIA', 80, 22, 6),
    # 'Argentina': ('ARGENTINA', -65, -35, 5.5),
    # 'Mexico':    ('MEXICO', -103, 24, 5),
    # 'Japan':     ('JAPAN', 143, 37, 5),
    # 'France':    ('FRANCE', 2.5, 47, 4.5),
    # 'Germany':   ('GERMANY', 10, 51, 4.5),
    # 'Spain':     ('SPAIN', -3.5, 40, 4.5),
    # 'United Kingdom': ('UK', -3, 55, 4),
    # 'Norway':    ('NORWAY', 12, 64, 4.5),
    # 'Portugal':  ('PORTUGAL', -8, 39.5, 3.5),
    # 'New Zealand': ('NEW\nZEALAND', 173, -42, 3.5),
    # 'Colombia':  ('COLOMBIA', -73, 4, 4.5),
    # 'Peru':      ('PERU', -75, -10, 5),
    # 'Chile':     ('CHILE', -71, -30, 4),
    # 'Bolivia':   ('BOLIVIA', -65, -17, 4),
    # 'Venezuela': ('VENEZUELA', -66, 7.5, 4),
    # 'Nigeria':   ('NIGERIA', 8, 9, 4.5),
    # 'South Africa': ('SOUTH\nAFRICA', 25, -30, 4),
    # 'Mali':      ('MALI', -2, 17, 4.5),
    # 'Niger':     ('NIGER', 9, 17, 4.5),
    # 'Chad':      ('CHAD', 19, 15, 4.5),
    # 'Dem. Rep. Congo': ('DR CONGO', 24, -3, 4),
    # 'Angola':    ('ANGOLA', 18, -12, 4.5),
    # 'Cuba':      ('CUBA', -79, 22, 3.5),
    # # 'Belarus':   ('BELARUS', 28, 53.5, 3.5),
    # # 'Kazakhstan': ('KAZAKHSTAN', 67, 48, 5),
}

In [12]:
# ════════════════════════════════════════════════════════════════════
# STEP 3: Load data, build map (usually no edits needed below)
# ════════════════════════════════════════════════════════════════════

GEOJSON_PATH = 'countries_10m.geojson'   # path to your converted geojson
OUTPUT_PATH  = 'world_map.svg'
DPI          = 1200
FIGSIZE      = (22, 11)

# Load and fix geometries
world = gpd.read_file(GEOJSON_PATH)
world['geometry'] = world['geometry'].apply(fix_geometry)
world = world[~world['geometry'].is_empty]
world = world[world['name'] != 'Antarctica']

# Assign categories
c2cat = {}
for cat, countries in category_map.items():
    for c in countries:
        c2cat[c] = cat
world['category'] = world['name'].map(c2cat)

# ── Draw ──
fig, ax = plt.subplots(1, 1, figsize=FIGSIZE)

# Gray background
world[world['category'].isna()].plot(
    ax=ax, color='#F0F0F0', edgecolor='#AAAAAA', linewidth=0.3)

# Colored groups
for cat_name, color in colors.items():
    print(f"Drawing category '{cat_name}'...")
    subset = world[world['category'] == cat_name]
    if len(subset) == 0:
        continue

    subset.plot(ax=ax, color=color, edgecolor='#333333', linewidth=0.5)

# Labels
for name, (label, lx, ly, fs) in label_map.items():
    ax.text(lx, ly, label, fontsize=fs, fontweight='bold',
            ha='center', va='center', color='#222222', alpha=0.7,
            fontfamily='sans-serif')

dataset_count = [35, 16, 14, 5, 4, 4, 3, 3, 3]

# Legend
legend_items = [
    mpatches.Patch(facecolor=colors[cat], edgecolor='#333333', linewidth=0.5,
                   label=f'{cat} ({count})')
    for cat, count in zip(colors.keys(), dataset_count)
]
ax.legend(handles=legend_items, loc='lower left', fontsize=20,
          frameon=True, framealpha=0.95, edgecolor='none',
          bbox_to_anchor=(0.005, 0.02), handlelength=1.8,
          handleheight=1.3, borderpad=1.0, labelspacing=0.6)

ax.set_xlim(-180, 180)
ax.set_ylim(-58, 85)
ax.set_facecolor('white')
ax.set_axis_off()
fig.patch.set_facecolor('white')

plt.tight_layout(pad=0.5)
plt.savefig(OUTPUT_PATH, dpi=DPI, bbox_inches='tight',
            facecolor='white', pad_inches=0.1)
plt.close()
print(f"Saved to {OUTPUT_PATH}")

Drawing category 'English'...
Drawing category 'Chinese'...
Drawing category 'Spanish'...
Drawing category 'Japanese'...
Drawing category 'German'...
Drawing category 'Russian'...
Drawing category 'French'...
Drawing category 'Norwegian'...
Drawing category 'Portuguese'...
Saved to world_map.svg


In [20]:
for k, v in world["name"].items(): 
    print(v)

Zimbabwe
Zambia
Yemen
Vietnam
Venezuela
Vatican
Vanuatu
Uzbekistan
Uruguay
Micronesia
Marshall Is.
N. Mariana Is.
U.S. Virgin Is.
Guam
American Samoa
Puerto Rico
United States of America
S. Geo. and the Is.
Br. Indian Ocean Ter.
Saint Helena
Pitcairn Is.
Anguilla
Falkland Is.
Cayman Is.
Bermuda
British Virgin Is.
Turks and Caicos Is.
Montserrat
Jersey
Guernsey
Isle of Man
United Kingdom
United Arab Emirates
Ukraine
Uganda
Turkmenistan
Turkey
Tunisia
Trinidad and Tobago
Tonga
Togo
Timor-Leste
Thailand
Tanzania
Tajikistan
Taiwan
Syria
Switzerland
Sweden
eSwatini
Suriname
S. Sudan
Sudan
Sri Lanka
Spain
South Korea
South Africa
Somalia
Somaliland
Solomon Is.
Slovakia
Slovenia
Singapore
Sierra Leone
Seychelles
Serbia
Senegal
Saudi Arabia
São Tomé and Principe
San Marino
Samoa
St. Vin. and Gren.
Saint Lucia
St. Kitts and Nevis
Rwanda
Russia
Romania
Qatar
Portugal
Poland
Philippines
Peru
Paraguay
Papua New Guinea
Panama
Palau
Pakistan
Oman
Norway
North Korea
Nigeria
Niger
Nicaragua
New Zealan